In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np
import pandas as pd
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter

In [55]:
with open('girls-names.csv') as f:
    content = f.readlines()
print(len(content))
content = ' '.join( [c.strip()+'.' for c in content])
content[:100]

3759


'Aadhya. Aadya. Aaira. Aairah. Aaliyah. Aanya. Aaradhya. Aarna. Aarvi. Aarya. Aashvi. Aavya. Aayat. A'

In [56]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(device)

mps


In [57]:
torch.random.manual_seed(1234)

In [59]:
allowed_chars = set(string.ascii_letters + string.digits + ". ")

content = ''.join([c for c in content if c in allowed_chars])

chars = set(sorted(list(content)))
vocab_size = len(chars)
print(f'vocab_size is {vocab_size}')
i_to_c = {k:v for k, v in enumerate(chars)}
c_to_i = {k:v for v, k in i_to_c.items()}

encode = lambda s: [c_to_i[c] for c in s] # noqa: E731
decode = lambda s: ''.join([i_to_c[i] for i in s]) # noqa: E731
decode(encode(content[:100]))

encoded_content = torch.tensor(encode(content))
print(encoded_content.shape)
print(f'number of encoded chars {len(encoded_content):,}')
print(f'sample encoded content {encoded_content[:100]}')

vocab_size is 54
torch.Size([30060])
number of encoded chars 30,060
sample encoded content tensor([31, 20, 18,  1, 34, 20, 35, 40, 31, 20, 18, 34, 20, 35, 40, 31, 20,  8,
        53, 20, 35, 40, 31, 20,  8, 53, 20,  1, 35, 40, 31, 20, 30,  8, 34, 20,
         1, 35, 40, 31, 20, 11, 34, 20, 35, 40, 31, 20, 53, 20, 18,  1, 34, 20,
        35, 40, 31, 20, 53, 11, 20, 35, 40, 31, 20, 53, 42,  8, 35, 40, 31, 20,
        53, 34, 20, 35, 40, 31, 20,  6,  1, 42,  8, 35, 40, 31, 20, 42, 34, 20,
        35, 40, 31, 20, 34, 20, 15, 35, 40, 31])


In [60]:
context_length = 6
embedding_dim = 512


class Head(nn.Module):
    def __init__(self, head_dimension, n_embd):
        super().__init__()

        self.query = nn.Linear(n_embd, head_dimension, bias=False)
        self.key = nn.Linear(n_embd, head_dimension, bias=False)
        self.value = nn.Linear(n_embd, head_dimension, bias=False)
        
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
        

    def forward(self, x):
        B, T, C = x.shape
        print("In forward: ", B, T, C)
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1)*(C**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        out =  wei @ v

        return out
    
    
class TransformerNameGenerator(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd):
        super().__init__()
        self.n_embd = n_embd
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_length, n_embd)
        self.head_model = Head(self.n_embd, self.n_embd)
        self.llm_model = nn.Linear(n_embd, vocab_size)
            

    def forward(self, idx):
        B, T = idx.shape
 
        token_embedding = self.token_embedding(idx) # B, T, embd_n
 
        pos = torch.arange(T)
 
        positional_embeddings = self.position_embedding(pos) # T, nemb
 
        x = token_embedding + positional_embeddings

        x = self.head_model(x) # B, T, n_embd
        logits = self.llm_model(x)
 
        return logits


In [12]:
model = TransformerNameGenerator(len(chars), context_length, embedding_dim)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [16]:
from torch.utils.data import TensorDataset, DataLoader

xs = []
ys = []
for i in range(0, len(encoded_content)-context_length):
    x_chunk = encoded_content[i:i+context_length]
    y_chunk = encoded_content[i+1:i+context_length+1]

    xs.append(x_chunk)
    ys.append(y_chunk)

X = torch.stack(xs)
Y = torch.stack(ys)

dataset = TensorDataset(X, Y)

loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [233]:
for xb, yb in loader:
    logits = model.forward(xb)
    B, T, C = logits.shape
    logits = logits.view(B*T, C)
    targets = yb.view(B*T)
    loss = F.cross_entropy(logits, targets)
    print(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

3.971930742263794
9.353391647338867
13.17187786102295
11.473908424377441
11.340167999267578
9.886701583862305
7.713230609893799
8.585976600646973
6.833381652832031
8.574597358703613
8.349566459655762
7.520266056060791
7.072554111480713
6.2999420166015625
7.54346227645874
6.145204544067383
7.774266719818115
6.292397975921631
6.5742506980896
6.282945156097412
5.522074222564697
6.049063205718994
5.0443925857543945
5.504887104034424
5.4895548820495605
5.297463893890381
5.388692855834961
4.8359551429748535
4.506224632263184
4.249347686767578
4.972417831420898
5.146539211273193
4.350394248962402
4.938486576080322
4.429964542388916
4.16636323928833
4.370264530181885
3.7479021549224854
3.9251010417938232
4.170289516448975
4.190051555633545
3.841796875
3.847409248352051
3.836397409439087
3.729637861251831
3.74479603767395
3.359759569168091
4.021187782287598
3.52645206451416
3.40559983253479
3.542821168899536
3.3483896255493164
3.6423912048339844
3.1934783458709717
3.7783870697021484
3.845089673

In [235]:
for xb, yb in loader:
    print(xb.shape)
    break

torch.Size([32, 6])


In [251]:
decode(model.forward(torch.tensor(encode('Anthon')).view(1,-1)).argmax(dim=-1).tolist()[0])

'mae.na'